# Stage 4 — Bib OCR Evaluation

Evaluate PaddleOCR bib number reading on track crops.

**What this notebook covers:**
- Running `pipeline.ocr.BibOCR` on sample frames (every 5th frame)
- Visualising OCR crops with predicted bib numbers
- `resolve_bibs` majority-vote aggregation
- Accuracy check if ground truth bib map is available

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

VIDEO_PATH = Path('../data/raw_footage/sample_clip.mp4')
TARGET_FPS = 15
MAX_FRAMES = 60
JOB_ID     = 'notebook-eval-detection'

# Optional ground truth: {track_id: expected_bib_number}
GROUND_TRUTH = {}   # e.g. {1: 7, 2: 14, 3: 22}

from pipeline.cache import PipelineCache
cache = PipelineCache(job_id=JOB_ID, cache_root=Path('../data/cache'))
print("Stages cached:", cache.summary().get('stages_cached', []))

In [ ]:
from pipeline.ingest import extract_frames

frames, frame_indices, timestamps = [], [], []
for fi, frame, ts in extract_frames(str(VIDEO_PATH), target_fps=TARGET_FPS):
    frames.append(frame); frame_indices.append(fi); timestamps.append(ts)
    if MAX_FRAMES and len(frames) >= MAX_FRAMES: break

if cache.has('track'):
    all_tracks = cache.load_tracks()[:len(frames)]
else:
    from pipeline.detect import PersonDetector
    from pipeline.track import PersonTracker
    detector = PersonDetector(); tracker = PersonTracker()
    all_tracks = []
    for i, frame in enumerate(frames):
        dets = detector.detect(frame)
        for d in dets: d.frame_idx = frame_indices[i]
        all_tracks.append(tracker.update(dets, frame_idx=frame_indices[i]))
    cache.save_tracks(all_tracks)

print(f"Frames: {len(frames)}  Tracks: {len(set(t.track_id for ft in all_tracks for t in ft))}")

In [ ]:
from pipeline.ocr import BibOCR, resolve_bibs

ocr = BibOCR()
frame_readings = []

for i, (frame, tracks) in enumerate(zip(frames, all_tracks)):
    if i % 5 == 0:
        readings = ocr.read_frame(frame, tracks)
        frame_readings.append(readings)
        if readings:
            print(f"  t={timestamps[i]:.1f}s  readings={readings}")

resolved = resolve_bibs(frame_readings)
cache.save_ocr(frame_readings, resolved)

print(f"\nResolved bibs (track_id → (bib, confidence)):")
for tid, (bib, conf) in sorted(resolved.items()):
    match = "" if not GROUND_TRUTH else (
        "✓" if GROUND_TRUTH.get(tid) == bib else f"✗ (expected {GROUND_TRUTH.get(tid)})")
    print(f"  Track {tid:2d} → bib={bib}  conf={conf:.2f}  {match}")

In [ ]:
# ── Visualise OCR crops with predictions ─────────────────────────────────────
from pipeline.ocr import BibOCR   # reuse for crop utility

def extract_crop(frame, bbox, expand=1.3):
    x1, y1, x2, y2 = bbox
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    hw, hh = (x2 - x1) / 2 * expand, (y2 - y1) / 2 * expand
    x1c = max(0, int(cx - hw)); y1c = max(0, int(cy - hh))
    x2c = min(frame.shape[1], int(cx + hw)); y2c = min(frame.shape[0], int(cy + hh))
    return frame[y1c:y2c, x1c:x2c]

sample_frame = frames[min(10, len(frames)-1)]
sample_tracks = all_tracks[min(10, len(all_tracks)-1)]

if sample_tracks:
    n = len(sample_tracks)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    if n == 1: axes = [axes]
    for ax, track in zip(axes, sample_tracks):
        crop = extract_crop(sample_frame, track.bbox)
        bib, conf = resolved.get(track.track_id, (None, 0.0))
        ax.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
        ax.set_title(f"ID{track.track_id}\nbib={bib}  c={conf:.2f}", fontsize=10)
        ax.axis('off')
    plt.suptitle('OCR Crops — Bib Predictions', fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()
else:
    print("No tracks in sample frame.")